# Part 2

### Config & schema creation

In [0]:
dbutils.widgets.text("catalog", "de_assessment_dev")
CATALOG = dbutils.widgets.get("catalog")

spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
# Imports
import time
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, ArrayType
)


### Create silver shows

In [0]:
bronze_shows = spark.table(f"{CATALOG}.bronze.bronze_shows")

rating_schema  = StructType([StructField("average", DoubleType())])
network_schema = StructType([
    StructField("id",      IntegerType()),
    StructField("name",    StringType()),
    StructField("country", StructType([StructField("name", StringType())]))
])
genres_schema = ArrayType(StringType())

silver_shows = (
    bronze_shows
    .withColumn("_rating",  F.from_json("rating",  rating_schema))
    .withColumn("_network", F.from_json("network", network_schema))
    .withColumn("_genres",  F.from_json("genres",  genres_schema))
    .select(
        F.col("id").cast(IntegerType()).alias("show_id"),
        F.col("name").alias("show_name"),
        F.col("language"),
        F.col("status"),
        F.col("runtime").cast(IntegerType()),
        F.col("averageRuntime").cast(IntegerType()).alias("average_runtime"),
        F.col("premiered"),
        F.col("ended"),
        F.col("_rating.average").alias("rating_average"),
        F.col("_network.name").alias("network_name"),
        F.col("_network.country.name").alias("network_country"),
        F.col("_genres").alias("genres"),
    )
    .withColumn("genre", F.explode_outer("genres"))
    .drop("genres")
    .filter(F.col("show_id").isNotNull())
    .fillna({"language": "Unknown", "status": "Unknown",
             "network_name": "Unknown", "network_country": "Unknown", "genre": "Unknown"})
    .dropDuplicates(["show_id", "genre"])
)

silver_shows.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.silver.silver_shows")

print("silver_shows:", silver_shows.count(), "rows")

### Create silver episodes

In [0]:
bronze_episodes = spark.table(f"{CATALOG}.bronze.bronze_episodes")

silver_episodes = (
    bronze_episodes
    .select(
        F.col("id").cast(IntegerType()).alias("episode_id"),
        F.col("show_id").cast(IntegerType()).alias("show_id"),
        F.col("name").alias("episode_name"),
        F.col("season").cast(IntegerType()),
        F.col("number").cast(IntegerType()).alias("episode_number"),
        F.col("airdate"),
        F.col("runtime").cast(IntegerType()),
    )
    .filter(F.col("episode_id").isNotNull())
    .fillna({"episode_name": "Unknown", "airdate": "Unknown"})
    .filter(F.col("runtime") > 0)
    .dropDuplicates(["episode_id"])
)

silver_episodes.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("season") \
    .saveAsTable(f"{CATALOG}.silver.silver_episodes")

print("silver_episodes:", silver_episodes.count(), "rows")

### Create silver cast

In [0]:
bronze_cast = spark.table(f"{CATALOG}.bronze.bronze_cast")

person_schema    = StructType([StructField("id", IntegerType()), StructField("name", StringType())])
character_schema = StructType([StructField("id", IntegerType()), StructField("name", StringType())])

silver_cast = (
    bronze_cast
    .withColumn("_person",    F.from_json("person",    person_schema))
    .withColumn("_character", F.from_json("character", character_schema))
    .select(
        F.col("show_id").cast(IntegerType()).alias("show_id"),
        F.col("_person.id").cast(IntegerType()).alias("person_id"),
        F.col("_person.name").alias("cast_name"),
        F.col("_character.name").alias("character_name"),
    )
    .filter(F.col("person_id").isNotNull())
    .fillna({"cast_name": "Unknown", "character_name": "Unknown"})
    .dropDuplicates(["show_id", "person_id"])
)

silver_cast.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.silver.silver_cast")

print("silver_cast:", silver_cast.count(), "rows")

### Fact table

In [0]:
import time

shows    = spark.table(f"{CATALOG}.silver.silver_shows")
episodes = spark.table(f"{CATALOG}.silver.silver_episodes")
cast     = spark.table(f"{CATALOG}.silver.silver_cast")

# Broadcast cast (small table) — avoids shuffle
cast_broadcast = F.broadcast(cast.select("show_id", "cast_name", "character_name"))

t0 = time.time()

fact_show_data = (
    shows.alias("s")
    .join(episodes.alias("e"), F.col("s.show_id") == F.col("e.show_id"), "inner")
    .join(cast_broadcast.alias("c"), F.col("s.show_id") == F.col("c.show_id"), "left")
    .select(
        F.col("s.show_id"),
        F.col("s.show_name"),
        F.col("s.language"),
        F.col("s.genre"),
        F.col("e.season"),
        F.col("e.episode_name"),
        F.col("e.airdate"),
        F.col("e.runtime"),
        F.col("c.cast_name"),
        F.col("c.character_name"),
    )
    .fillna({"cast_name": "Unknown", "character_name": "Unknown"})
    .dropDuplicates(["show_id", "season", "episode_name", "cast_name"])
)

fact_show_data.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("season") \
    .saveAsTable(f"{CATALOG}.silver.fact_show_data")

print(f"fact_show_data: {fact_show_data.count()} rows in {time.time()-t0:.1f}s")

In [0]:
import time
from pyspark.sql.types import IntegerType

shows    = spark.table(f"{CATALOG}.silver.silver_shows")
episodes = spark.table(f"{CATALOG}.silver.silver_episodes")
cast     = spark.table(f"{CATALOG}.silver.silver_cast")

# ── BEFORE: naive join, no broadcast, no salting ─────────────────────────────
t0 = time.time()
naive = (
    shows.alias("s")
    .join(episodes.alias("e"), "show_id", "inner")
    .join(cast.alias("c"),     "show_id", "left")
    .select("s.show_id", "s.show_name", "e.season", "e.episode_name", "c.cast_name")
)
naive_count = naive.count()
t_naive = time.time() - t0
print(f"BEFORE (naive join):          {naive_count} rows in {t_naive:.2f}s")

# ── AFTER: broadcast cast + salted show/episode join ─────────────────────────
# Salting technique: replicate small table (shows) across N salt buckets,
# assign random salt to large table (episodes) to distribute skewed keys
# across partitions evenly.
SALT_BUCKETS = 8

cast_b = F.broadcast(cast.select("show_id", "cast_name"))

# Small table — replicate across all salt buckets
shows_salted = shows.withColumn(
    "salt", F.explode(F.array([F.lit(i) for i in range(SALT_BUCKETS)]))
)
# Large table — random salt distributes skewed keys across partitions
episodes_salted = episodes.withColumn(
    "salt", (F.rand() * SALT_BUCKETS).cast(IntegerType())
)

t0 = time.time()
optimized = (
    shows_salted.alias("s")
    .join(episodes_salted.alias("e"),
          (F.col("s.show_id") == F.col("e.show_id")) & (F.col("s.salt") == F.col("e.salt")), "inner")
    .join(cast_b.alias("c"), F.col("s.show_id") == F.col("c.show_id"), "left")
    .select("s.show_id", "s.show_name", "e.season", "e.episode_name", "c.cast_name")
)
opt_count = optimized.count()
t_opt = time.time() - t0
print(f"AFTER  (broadcast + salting): {opt_count} rows in {t_opt:.2f}s")
print(f"Speedup: {t_naive/t_opt:.1f}x faster")

# Row counts should match — salting must not lose data
assert naive_count == opt_count, f"Data loss! naive={naive_count}, optimized={opt_count}"

### Optimize ZORDER

In [0]:
import time

for table, zcols in [
    (f"{CATALOG}.silver.silver_shows",    "show_id"),
    (f"{CATALOG}.silver.silver_episodes", "show_id"),
    (f"{CATALOG}.silver.silver_cast",     "show_id"),
    (f"{CATALOG}.silver.fact_show_data",  "show_id, airdate"),
]:
    try:
        t0 = time.time()
        spark.sql(f"OPTIMIZE {table} ZORDER BY ({zcols})")
        print(f"OPTIMIZE {table} — {time.time()-t0:.1f}s")
    except Exception as e:
        print(f"Warning: OPTIMIZE failed for {table}: {e}")

In [0]:
# NOT NULL constraints — enforce primary keys can never be null
for tbl, cols in [
    ("silver_shows",    ["show_id"]),
    ("silver_episodes", ["episode_id", "show_id"]),
    ("silver_cast",     ["show_id", "person_id"]),
    ("fact_show_data",  ["show_id"]),
]:
    for col in cols:
        try:
            spark.sql(f"ALTER TABLE {CATALOG}.silver.{tbl} ALTER COLUMN {col} SET NOT NULL")
            print(f"OK   {tbl}.{col} SET NOT NULL")
        except Exception as e:
            if "already" in str(e).lower():
                print(f"SKIP {tbl}.{col} — already NOT NULL")
            else:
                print(f"WARN {tbl}.{col} — {e}")

# CHECK constraints — enforce business rules at the storage level
constraints = [
    ("silver_episodes", "valid_runtime",    "runtime > 0"),
    ("silver_episodes", "valid_season",     "season >= 1"),
    ("silver_shows",    "valid_show_id",    "show_id > 0"),
    ("silver_cast",     "valid_person_id",  "person_id > 0"),
]

for tbl, name, expr in constraints:
    try:
        spark.sql(f"ALTER TABLE {CATALOG}.silver.{tbl} ADD CONSTRAINT {name} CHECK ({expr})")
        print(f"OK   {tbl}: added CHECK {name} ({expr})")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"SKIP {tbl}: {name} already exists")
        else:
            print(f"WARN {tbl}: {name} — {e}")

print("\nSchema enforcement applied.")

### Verify

In [0]:
for tbl in ["silver_shows", "silver_episodes", "silver_cast", "fact_show_data"]:
    print(f"{tbl}: {spark.table(f'{CATALOG}.silver.{tbl}').count()} rows")

spark.table(f"{CATALOG}.silver.fact_show_data").limit(10).display()

In [0]:
# # Data Quality Assertions
# errors = []

# for tbl, pk in [
#     ("silver_shows",    ["show_id", "genre"]),
#     ("silver_episodes", ["episode_id"]),
#     ("silver_cast",     ["show_id", "person_id"]),
#     ("fact_show_data",  ["show_id", "season", "episode_name", "cast_name"]),
# ]:
#     try:
#         df = spark.table(f"{CATALOG}.silver.{tbl}")
#         total   = df.count()
#         deduped = df.dropDuplicates(pk).count()
#         nulls   = df.filter(F.col(pk[0]).isNull()).count()

#         if total != deduped:
#             errors.append(f"FAIL {tbl}: {total - deduped} duplicate rows on {pk}")
#         else:
#             print(f"OK   {tbl}: no duplicates on {pk} ({total} rows)")

#         if nulls > 0:
#             errors.append(f"FAIL {tbl}: {nulls} null values in {pk[0]}")
#         else:
#             print(f"OK   {tbl}: no nulls in {pk[0]}")
#     except Exception as e:
#         errors.append(f"FAIL {tbl}: could not validate — {e}")

# if errors:
#     print("\nData quality issues found:")
#     for err in errors:
#         print(f"  {err}")
# else:
#     print("\nAll data quality checks passed.")

In [0]:
import json

expected = {
    "silver_shows":    {"show_id", "show_name", "language", "genre"},
    "silver_episodes": {"episode_id", "show_id", "season", "runtime"},
    "silver_cast":     {"show_id", "person_id", "cast_name", "character_name"},
    "fact_show_data":  {"show_id", "show_name", "genre", "season", "episode_name", "cast_name"},
}

issues = []
counts = {}
for tbl, cols in expected.items():
    try:
        actual = set(spark.table(f"{CATALOG}.silver.{tbl}").columns)
        missing = cols - actual
        if missing:
            issues.append(f"{tbl} missing columns: {missing}")
        cnt = spark.table(f"{CATALOG}.silver.{tbl}").count()
        counts[tbl] = cnt
        if cnt == 0:
            issues.append(f"{tbl} is empty")
    except Exception as e:
        issues.append(f"{tbl}: could not read table — {e}")
        counts[tbl] = -1

if issues:
    msg = "; ".join(issues)
    print(f"Silver validation FAILED: {msg}")
    dbutils.notebook.exit(json.dumps({"status": "FAILED", "reason": msg, "counts": counts}))

print("Silver validation passed:", counts)
dbutils.notebook.exit(json.dumps({"status": "OK", "counts": counts}))